# Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Loadig and inspecting Dataset

In [ ]:
df = pd.read_csv('/Users/festusattornelson/Documents/Projects/Python MSIT/car_data.csv')
df.head(2)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Checkimg and dropping duplicate rows
df.duplicated().sum()

In [ ]:
#df = df.drop_duplicates(keep='first').reset_index(drop=True)
#df.shape

In [ ]:
# Checking for missing values
missing_values = df.isna().sum()
percent_missing_values = (df.isna().sum()/df.shape[0])*100
missing_values_summary= pd.DataFrame({'missing values': missing_values, 'missing_values (%)': percent_missing_values})
missing_values_summary

In [ ]:
# Cleansing columns for uniformity
#df.columns = [col.replace(' ', '') for col in df.columns]

In [ ]:
# convert year to datetime
#df['Year'] = pd.to_datetime(df['Year'], format='%Y')

In [ ]:
#df = df.map(lambda row: row.lower() if isinstance(row, str) else row)

In [ ]:
def initial_clean_data(data):
    """
      Cleans a given DataFrame by:
    - Removing duplicate rows
    - Standardizing text columns (trimming spaces, converting to lowercase)
    - Filter data by year >= 1995

    Parameters:
    df (pd.DataFrame): The input DataFrame to be cleaned

    Returns:
    pd.DataFrame: The cleaned DataFrame
    
    """
    df = data.copy()

    #Remove duplicates
    df = df.drop_duplicates(keep='first').reset_index(drop=True)

    #Filter columns based on year >= 1995
    df = df.loc[df.Year >= 1995].reset_index(drop=True)

    #Convert year to datetime
    df['Year'] = pd.to_datetime(df['Year'], format='%Y')

    #Format text columns
    df.columns = [col.replace(' ', '').replace('_', '') for col in df.columns]

    df = df.map(lambda row: row.lower() if isinstance(row, str) else row)

    
    return df

In [ ]:
# Clean the data
df_car = initial_clean_data(data=df)

print("Initial clean dataframe:")
df_car.head()

In [ ]:
df_car.shape

In [ ]:
df_car.info()

In [ ]:
#categorical and numerical columns
categorical_columns = []
numerical_columns = []

for col in df_car.columns:
  if df_car[col].dtype == 'object':
    categorical_columns.append(col)
  else:
    numerical_columns.append(col)

In [ ]:
categorical_columns

In [ ]:
numerical_columns

# 1. Data Cleansing

In [ ]:
#Handling missing values
df_car.isna().sum()

In [ ]:
#df_car.loc[(df_car.EngineCylinders.isna())]#.head()

In [ ]:
#Electric cars don't have engine cylinders = 0
df_car.loc[(df_car.TransmissionType == 'direct_drive') & (df_car.EngineFuelType == 'electric'), 'EngineCylinders'] = 0 

In [ ]:
#Mazda RX-7or8 has rotary (wankel) engine hence 0 conventional cylinders
df_car.loc[(df_car.Model == 'rx-7') | (df_car.Model == 'rx-8'), 'EngineCylinders'] = 0 

In [ ]:
#Checking missing values for "EngineFuelType"
#df_car.loc[(df_car.Make == 'suzuki') & (df_car.Model == 'verona') & (df_car.EngineHP == 155.0) & (df_car.TransmissionType == 'automatic')] 

In [ ]:
# Fill missing EngineFuelType using vales from similar make, model, EngineHP, Cylinders and transmission type
df_car['EngineFuelType'] = df_car['EngineFuelType'].fillna(df_car.loc[10033, 'EngineFuelType'])

In [ ]:
df_car.loc[(df_car.Make == 'fiat') & (df_car.Model == '500e') & (df_car.Year == 2015) & (df_car.EngineFuelType == 'electric'), 'EngineHP'] = 111

In [ ]:
df_car.loc[(df_car.Make == 'lincoln') & (df_car.Model == 'continental') & (df_car.Year == 2017), 'EngineHP'] = 305

In [ ]:
df_car.loc[(df_car.Make.str.lower() == 'kia') &(df_car.Model.str.lower() == 'soul ev') &
    (df_car.Year.isin([2015, 2016])) &(df_car.EngineFuelType.str.lower() == 'electric'),'EngineHP'] = 109

In [ ]:
#df_car.loc[(df_car.NumberofDoors.isna())]

In [ ]:
#Fill the missing values of marketcategory based on mode or make
df_car['MarketCategory'] = df_car.groupby(['Make', 'Model'])['MarketCategory'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown')
)

In [ ]:
df_car['NumberofDoors'] = df_car['NumberofDoors'].fillna(df_car.loc[6188, 'NumberofDoors'])

In [ ]:
df_car.loc[(df_car.Make == 'tesla')].head(5)

In [ ]:
df_car.isna().sum()

# 2. Feature Engineering

# Exploratory ata Analysis (EDA)